
# Reprodutibilidade e diagnóstico das dependências do PySITS

## 1. Objetivo

O notebook tem como objetivo a reprodução da instalação do PySITS no Geolab, tomando como referência o repositório original `mlops-lulc`.

Assim, vamos identificar problemas de compatibilidade entre dependências, registrar erros encontrados durante a instalação e execução e documentar as modificações necessárias para viabilizar o funcionamento do PySITS.

## 2. Ambiente experimental

O experimento é conduzido no ambiente Geolab, usando o VS Code como interface de desenvolvimento.

Foi feito um *fork* do repositório original, permitindo manter um espaço independente para investigar a reprodutibilidade, sem modificar diretamente o projeto de referência.


## 3. Metodologia

A investigação segue os passos:

1. Caracterização do ambiente computacional disponível.
2. Reprodução do procedimento original de instalação.
3. Identificação e registro de erros e incompatibilidades.
4. Ajuste das dependências incompatíveis.
5. Validação da instalação e funcionalidade do PySITS.


## 4. Geolab

### 4.1. Configuração original do ambiente

Registramos as versões das dependências principais envolvendo a instalação do PySITS.

- Python: 3.12.16 (main, Oct 26 2026, 11:37:43) [GCC 15.3.0]
- pysits: 1.5.4
- pandas: 3.1.1
- rpy2: 3.6.9
- pyarrow: 25.0.1

In [1]:
import sys
import importlib.metadata as metadata

print("Python:", sys.version)

packages = ["pysits", "pandas", "rpy2", "pyarrow"]

for package in packages:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

Python: 3.12.16 (main, Oct 26 2026, 11:37:43) [GCC 15.3.0]
pysits: 1.5.4
pandas: 3.1.1
rpy2: 3.6.9
pyarrow: 25.0.1


### 4.2. Erro inicial de importação

Depois de instalar o PySITS e suas dependências, tentamos a importação do pacote para verificar a instalação.

In [2]:
from pysits import *
print("PySITS OK")

Error importing in API mode: ModuleNotFoundError("No module named '_rinterface_cffi_api'")
Trying to import in ABI mode.


ImportError: cannot import name 'Self' from 'pandas._typing' (/home/jovyan/.conda/envs/pysits/lib/python3.12/site-packages/pandas/_typing.py)

### 4.3. Análise do erro

A falha mostrou duas mensagens relevantes:

1. Error importing in API mode: 
    ModuleNotFoundError("No module named `_rinterface_cffi_api`")

2. ImportError: cannot import name `Self` from `pandas._typing`
.

A falha ocorreu durante `pysits.models.frame`.


### 4.4. Correção da dependência

Após analisar o repositório oficial do PySITS no GitHub, identificamos que a versão mínima de pandas indicada é a 2.2.3.

Diante da incompatibilidade identificada na versão inicialmente instalada (pandas 3.1.1), realizamos a instalação manual da versão 2.2.3 por meio do seguinte comando, executado no terminal do ambiente Conda:

`conda install pandas=2.2.3`

A alteração foi realizada sem recriar o ambiente Conda, mantendo as demais dependências instaladas.

In [1]:
import pandas as pd

print("Pandas version:", pd.__version__)

import pysits

print("PySITS OK")

Pandas version: 2.2.3


Error importing in API mode: ModuleNotFoundError("No module named '_rinterface_cffi_api'")
Trying to import in ABI mode.


PySITS OK


### 4.5. Validação da importação do PySITS

Após mudar a versão do pandas para 2.2.3 e reiniciar o kernel, repetimos o teste de importação do PySITS.

A saída mostrou que o rpy2 não conseguiu inicializar no modo API, pois não há um módulo de `_rinterface_cffi_api` disponível. Como alternativa, o rpy2 tentou fazer a inicialização no modo ABI.

Apesar do aviso, o pacote PySITS foi importado com sucesso, conforme indica a mensagem `PySITS OK`.

Portanto, o ajuste na versão do pandas resolveu o erro de importação anterior. A indisponibilidade do modo API do rpy2 não impediu a importação do PySITS neste ambiente.

### 4.6. Checagem da integração Python-R

Embora o aviso relacionado ao modo API do rpy2 não tenha impedido a importação do PySITS, realizamos testes adicionais para verificar a integração entre Python e R, bem como a disponibilidade das dependências necessárias no ambiente.

In [ ]:
#Verificar o R pelo rpy2
import rpy2.robjects as ro

print(ro.r("R.version.string")[0])

R version 4.5.3 (2026-03-11)


In [4]:
#Verificar os pacotes R
ro.r("""
library(sits)
library(arrow)
print(packageVersion("sits"))
""")

[1] ‘1.5.4’


### 4.7. Teste de instalação e inspeção da interface

Como a documentação e os exemplos disponíveis no livro do SITS são voltados ao pacote R `sits`, não sendo necessariamente compatíveis com a interface Python do PySITS 1.5.4, optamos por realizar testes funcionais a partir da inspeção das funcionalidades disponíveis e da execução de operações do pacote.

Inicialmente, verificamos os metadados da instalação, as funções disponibilizadas pelo módulo e as coleções de dados reconhecidas pelo pacote. Em seguida, investigamos funções específicas para identificar possibilidades de execução de testes práticos.

In [10]:
#Metadados do pacote PySITS
%pip show pysits

Name: pysits
Version: 1.5.4
Summary: Python wrapper for the sits R package
Home-page: https://github.com/e-sensing/pysits
Author: 
Author-email: Gilberto Camara <gilberto.camara.inpe@gmail.com>, Felipe Carlos <efelipecarlos@gmail.com>, Felipe Carvalho <lipecaso@gmail.com>, Pedro Brito <pedro_brito1997@hotmail.com>, Rolf Simoes <rolfsimoes@gmail.com>
License: GPL-2.0-only
Location: /home/jovyan/.conda/envs/pysits/lib/python3.12/site-packages
Requires: geopandas, jinja2, matplotlib, pandas, pillow, pyarrow, rpy2, shapely
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [11]:
#Verificar as funções disponíveis
 
import inspect

functions = [
    name for name, obj in inspect.getmembers(pysits)
    if inspect.isfunction(obj)
]

print(f"Total de funções disponíveis: {len(functions)}")

for name in functions:
    print(name)

Total de funções disponíveis: 101
hparam
impute_linear
impute_mean
impute_mean_window
impute_median
load_samples
plot
r_package_dir
r_set_seed
read_rds
sits_accuracy
sits_accuracy_summary
sits_add_base_cube
sits_apply
sits_as_geopandas
sits_as_xarray
sits_bands
sits_bbox
sits_classify
sits_clean
sits_cluster_clean
sits_cluster_dendro
sits_cluster_frequency
sits_colors
sits_colors_qgis
sits_colors_reset
sits_colors_set
sits_colors_show
sits_combine_predictions
sits_confidence_sampling
sits_config
sits_config_show
sits_config_user_file
sits_cube
sits_cube_copy
sits_formula_linear
sits_formula_logref
sits_geo_dist
sits_get_class
sits_get_data
sits_get_probs
sits_impute
sits_kfold_validate
sits_label_classification
sits_labels
sits_labels_summary
sits_lightgbm
sits_lighttae
sits_list_collections
sits_merge
sits_mgrs_to_roi
sits_mixture_model
sits_mlp
sits_model_export
sits_mosaic
sits_patterns
sits_pred_features
sits_pred_normalize
sits_pred_references
sits_pred_sample
sits_predictors
sits

In [8]:
# Verificar as coleções disponíveis

collections = pysits.sits_list_collections(source="BDC")

print(collections)

BDC:
- CBERS-WFI-16D (CBERS-4/WFI)
- grid system: BDC-Large V2
- period: 2016 to 2024
- bands: NDVI EVI B13 B14 B15 B16 CLOUD
- opendata collection (requires access token)

- CBERS-WFI-8D (CBERS-4-4A/WFI)
- grid system: BDC-Large V2
- period: 2020 to 2024
- bands: NDVI EVI B13 B14 B15 B16 CLOUD
- opendata collection (requires access token)

- CBERS-MUX-2M (CBERS-4/MUX)
- grid system: BDC-Medium V2
- period: 2016 to 2024
- bands: NDVI EVI B05 B06 B07 B08 CLOUD
- opendata collection (requires access token)

- LANDSAT-OLI-16D (LANDSAT/OLI)
- grid system: BDC-Medium V2
- period: 2010 to 2024
- bands: NDVI EVI COASTAL BLUE GREEN RED NIR08 SWIR16 SWIR22 CLOUD
- opendata collection (requires access token)

- MOD13Q1-6.1 (TERRA/MODIS)
- grid system: STG
- period: 2000 to 2024
- bands: NDVI EVI BLUE RED NIR MIR CLOUD
- opendata collection (requires access token)

- MYD13Q1-6.1 (AQUA/MODIS)
- grid system: STG
- period: 2002 to 2024
- bands: NDVI EVI BLUE RED NIR MIR CLOUD
- opendata collection (

### 4.8. Verificação do funcionamento do PySITS

Após verificar a instalação e as funcionalidades disponíveis no PySITS 1.5.4, buscamos realizar um teste prático de recuperação de séries temporais.

O objetivo é verificar se o pacote consegue acessar e carregar dados de sensoriamento remoto, demonstrando seu funcionamento além da simples importação das bibliotecas e execução de funções informativas.

Para isso, buscamos utilizar uma coleção de dados disponível e compatível com a versão instalada do pacote.

In [ ]:
# Verificar documentação e funções
print(inspect.getsource(pysits.load_samples))
print(inspect.signature(pysits.load_samples))
print(inspect.getdoc(pysits.load_samples))

def load_samples(name: str, package: str, **kwargs) -> SITSTimeSeriesModel:
    """Load sits data from package.

    Args:
        name (str): Dataset name.

        package (str): Package name.

        **kwargs: Additional arguments to pass to the function.
    """
    # Load data from package
    load_data_from_package(name, package, **kwargs)

    # Load data from global environment
    data = load_data_from_global(name)

    # Return data
    return SITSTimeSeriesModel(data)

(name: str, package: str, **kwargs) -> pysits.models.data.ts.SITSTimeSeriesModel
Load sits data from package.

Args:
    name (str): Dataset name.

    package (str): Package name.

    **kwargs: Additional arguments to pass to the function.
def load_samples(name: str, package: str, **kwargs) -> SITSTimeSeriesModel:
    """Load sits data from package.

    Args:
        name (str): Dataset name.

        package (str): Package name.

        **kwargs: Additional arguments to pass to the function.
    """
    

In [16]:
import inspect
import pysits

print("sits_cube signature:")
print(inspect.signature(pysits.sits_cube))

print("\nsits_cube documentation:")
print(inspect.getdoc(pysits.sits_cube))

sits_cube signature:
(*args, **kwargs) -> pysits.models.data.cube.SITSCubeModel

sits_cube documentation:
(No documentation found for sits_cube)


In [17]:
print("sits_timeline signature:")
print(inspect.signature(pysits.sits_timeline))

print("\nsits_timeline documentation:")
print(inspect.getdoc(pysits.sits_timeline))


sits_timeline signature:
(*args, **kwargs) -> list[datetime.date]

sits_timeline documentation:
(No documentation found for sits_timeline)


In [21]:
from pysits import sits_cube, sits_timeline
# Define the analysis period
START_DATE = "2023-01-01"
END_DATE = "2023-12-31"

# Define a small region of interest (ROI)
ROI = {
    "xmin": -46.0,
    "xmax": -45.9,
    "ymin": -23.0,
    "ymax": -22.9
}

# Create a Sentinel-2 NDVI data cube
cube = sits_cube(
    source="BDC", # necessário senha para usuários externos, checar: https://brazildatacube.dpi.inpe.br/ 
    collection="SENTINEL-2-16D",
    bands=["NDVI"],
    start_date=START_DATE,
    end_date=END_DATE,
    roi=ROI,
    multicores=1
)

# Inspect the data cube
print(cube)

# Inspect the temporal coverage
print(sits_timeline(cube))


  |======================================================================| 100%
  source      collection   satellite sensor    tile       xmin       xmax  \
0    BDC  SENTINEL-2-16D  SENTINEL-2    MSI  030030  5792000.0  5897600.0   

        ymin       ymax                                                crs  \
0  8680000.0  8785600.0  PROJCRS["unknown",\n    BASEGEOGCRS["unknown",...   

                                           file_info  
0                            fid        date  ban...  
['2023-01-01', '2023-01-17', '2023-02-02', '2023-02-18', '2023-03-06', '2023-03-22', '2023-04-07', '2023-04-23', '2023-05-09', '2023-05-25', '2023-06-10', '2023-06-26', '2023-07-12', '2023-07-28', '2023-08-13', '2023-08-29', '2023-09-14', '2023-09-30', '2023-10-16', '2023-11-01', '2023-11-17', '2023-12-03', '2023-12-19']


## Análise de dependência e compatibilidade